# Prueba tecnica - Analista Jr de Inversiones

## 1. Manejo de datos
explicacion
logica de joins
secuencia de operaciones

## 2. Analisis de portafolios
formulas
codigo
resultados
interpretacion

## 3. Caso de automatizacion
explicacion del motor
codigo
tabla final
archivo de review

## 4. Agente ACH
investigacion
tabla comparativa
diseño del agente
codigo

### Contexto del problema

En este ejercicio se tienen cuatro tablas relacionadas con usuarios, asesores, portafolios y balances. La idea es construir una secuencia de operaciones que permita consultar los balances de ciertos portafolios bajo condiciones específicas.

Las tablas disponibles son:

- **Usuarios**
- **Portafolios**
- **Asesores**
- **Balances**

Cada una contiene información distinta, pero están conectadas entre sí por identificadores como `id_usuario`, `id_portafolio` y `id_asesor`.

El objetivo final es obtener los **balances de portafolios** que cumplan tres condiciones:

1. Pertenecen a usuarios cuyo asesor tiene el correo `insightswm@gmail.com`
2. Corresponden a un **tipo de portafolio específico**
3. Están dentro de un **rango de fechas determinado**

Para lograr esto es necesario combinar las tablas y luego aplicar los filtros adecuados.

### Enfoque utilizado

La forma más directa de resolver el problema es unir primero las tablas que contienen las relaciones entre usuarios, asesores y portafolios, y después incorporar la información de balances.

El proceso se puede dividir en tres pasos principales:

1. **Relacionar usuarios con sus portafolios**
2. **Relacionar esos usuarios con sus asesores**
3. **Traer los balances de cada portafolio**

Una vez que toda la información está en una misma tabla, se pueden aplicar los filtros necesarios sobre asesor, tipo de portafolio y rango de fechas.

También se eliminan algunos campos que no son necesarios para la consulta final para mantener la tabla más limpia.

### Secuencia de operaciones

A continuación se muestra una posible secuencia utilizando las funciones `join`, `filter` y `drop` indicadas en el enunciado.

1. Unir portafolios con usuarios utilizando el identificador de usuario.
join(Portafolios, Usuarios, id_usuario, Usuarios_portafolios)

2. Eliminar campos que no son necesarios para el análisis.
drop(Usuarios_portafolios, email & perfil_riesgo)

3. Unir la tabla resultante con la tabla de asesores para identificar quién administra cada usuario.
join(Usuarios_portafolios, Asesores, id_asesor, Portafolios_asesores)

4. Filtrar únicamente los usuarios cuyo asesor tenga el correo solicitado.  
filter(Portafolios_asesores, email = "insightswm@gmail.com")

5. Incorporar la información de balances de los portafolios.
join(Portafolios_asesores, Balances, id_portafolio, Portafolios_balances)

6. Filtrar los registros dentro del rango de fechas requerido.
filter(Portafolios_balances, fecha between fecha_inicio and fecha_fin)

7. Filtrar por el tipo de portafolio de interés.
filter(Portafolios_balances, tipo_portafolio = tipo_deseado)

### Resultado esperado

Después de aplicar esta secuencia, la tabla resultante contendría únicamente los registros de balances que cumplen las condiciones planteadas.

Las columnas más relevantes de la salida serían:

- `id_portafolio`
- `fecha`
- `balance`
- `tipo_portafolio`

Con esta información se puede consultar fácilmente el comportamiento de los portafolios administrados por el asesor indicado dentro del periodo seleccionado.

# Punto 2

### Contexto del ejercicio

En este punto se analiza una matriz de simulacion que contiene retornos anuales simulados para 55 activos. La matriz tiene dimension `[30000 x 55]`, donde cada fila representa una simulacion y cada columna corresponde a un activo.

Adicionalmente se tienen dos vectores de pesos llamados `P1` y `P2`, cada uno con dimension `[1 x 55]`. Estos vectores representan la composicion de dos portafolios distintos.

El objetivo es calcular para cada portafolio:

- el retorno esperado
- la volatilidad esperada

Este tipo de calculo es muy comun en analisis de portafolios porque permite entender la relacion entre riesgo y retorno dependiendo de como se distribuyen los pesos entre los activos.

### Enfoque utilizado

La idea es bastante directa.

Primero se calcula el retorno esperado de cada activo a partir de la matriz de simulacion. Como tenemos muchas simulaciones, el promedio de cada columna nos da una buena estimacion del retorno esperado.

Despues se calcula la matriz de covarianza entre los activos, que captura como se mueven los retornos entre si. Esta matriz es clave porque determina el riesgo conjunto del portafolio.

Con esa informacion se aplican dos formulas clasicas de teoria de portafolios:

Retorno esperado del portafolio:

\[
E(R_p) = P^T \mu
\]

donde:

- `P` es el vector de pesos
- `μ` es el vector de retornos esperados de los activos

Volatilidad esperada del portafolio:

\[
\sigma_p = \sqrt{P^T \Sigma P}
\]

donde:

- `Σ` es la matriz de covarianza de los retornos.

En el cuaderno se utiliza Python para realizar estos calculos utilizando la matriz de simulacion y los vectores de pesos proporcionados en el archivo.

In [1]:
from src.cargar_datos import cargar_matriz_simulacion, cargar_vector_pesos
from src.analisis_portafolios import (
    calcular_retorno_esperado_activos,
    calcular_matriz_covarianza,
    resumir_portafolio,
)

ruta_archivo = "datos/prueba.xlsx"

# cargar matriz de retornos simulados
matriz_simulacion = cargar_matriz_simulacion(
    ruta_archivo, "Matriz de Simulacion")

# cargar pesos de los portafolios
pesos_p1 = cargar_vector_pesos(ruta_archivo, "P1")
pesos_p2 = cargar_vector_pesos(ruta_archivo, "P2")

# calcular retorno esperado por activo
retornos_esperados = calcular_retorno_esperado_activos(matriz_simulacion)

# calcular matriz de covarianza
matriz_covarianza = calcular_matriz_covarianza(matriz_simulacion)

# resumir resultados de cada portafolio
resultado_p1 = resumir_portafolio(
    nombre_portafolio="P1",
    vector_pesos=pesos_p1,
    retornos_esperados=retornos_esperados,
    matriz_covarianza=matriz_covarianza,
)

resultado_p2 = resumir_portafolio(
    nombre_portafolio="P2",
    vector_pesos=pesos_p2,
    retornos_esperados=retornos_esperados,
    matriz_covarianza=matriz_covarianza,
)

resultado_p1, resultado_p2

({'portafolio': 'P1',
  'suma_pesos': 1.0,
  'activos_con_peso': 8,
  'retorno_esperado': 0.09119075722610004,
  'volatilidad_esperada': 0.20625330919875823},
 {'portafolio': 'P2',
  'suma_pesos': 1.0,
  'activos_con_peso': 8,
  'retorno_esperado': 0.05841911255639979,
  'volatilidad_esperada': 0.10107123485458074})

### Resultados obtenidos

Los resultados calculados para los dos portafolios fueron los siguientes:

| portafolio | retorno esperado | volatilidad esperada |
|---|---|---|
| P1 | 9.12% | 20.63% |
| P2 | 5.84% | 10.11% |

Antes de interpretar los resultados, se verifico tambien que los pesos de cada portafolio sumaran aproximadamente 1 y que solo algunos activos tuvieran peso distinto de cero. En ambos casos se observan **8 activos con peso positivo**, lo cual sugiere que los portafolios estan relativamente concentrados.

### Interpretacion

Al comparar los dos portafolios se observa una diferencia clara en el perfil de riesgo.

El portafolio P1 tiene un retorno esperado mas alto, cercano al 9.1%. Sin embargo, este mayor rendimiento viene acompañado de una volatilidad bastante mas elevada, alrededor del 20.6%. Esto indica que el portafolio esta expuesto a movimientos mas amplios en sus retornos.

Por otro lado, el portafolio P2 muestra un retorno esperado menor, cercano al 5.8%, pero tambien una volatilidad considerablemente mas baja, alrededor del 10.1%. En otras palabras, es un portafolio mas estable, aunque con menor potencial de retorno.

Este resultado refleja el trade off clasico entre riesgo y retorno en teoria de portafolios: estrategias que buscan mayor retorno suelen asumir tambien mayor volatilidad.

# Punto 3

### Contexto del problema

En este punto se tiene un conjunto de solicitudes de retiro junto con informacion adicional sobre las cuentas, actividad pendiente y destinos registrados.

El objetivo es construir una logica de decision que determine si cada solicitud debe:

- aprobarse (APPROVE)
- ponerse en revision manual (HOLD)
- rechazarse (REJECT)

La decision depende de varias reglas relacionadas con el estado de la cuenta, el riesgo AML, la disponibilidad de efectivo y posibles solicitudes duplicadas.

Este tipo de logica es bastante comun en sistemas financieros, donde se necesita automatizar decisiones operativas pero manteniendo ciertos controles de riesgo.

La idea es evaluar las reglas en un orden claro:

1. primero verificar condiciones de rechazo
2. si no se rechaza, evaluar condiciones de revision (hold)
3. si no se activa ninguna regla anterior, aprobar el retiro

Adicionalmente, cada decision debe tener un reason_code y un nivel de severity, lo cual permite priorizar los casos que requieren revision manual.

### logica de decision implementada

Para resolver el ejercicio se implemento un pequeño motor de decisiones que evalua cada solicitud de retiro de forma secuencial.

La logica sigue tres etapas principales.

Primero se revisan las condiciones que obligan a rechazar el retiro. Esto incluye casos como cuentas inactivas, kyc no verificado, montos invalidos, solicitudes duplicadas o cuentas con riesgo aml alto que no estan en lista blanca.

Si la solicitud no cae en ninguna condicion de rechazo, se evalúan las reglas que requieren revision manual. Entre ellas se incluyen situaciones donde el retiro podria dejar el balance disponible demasiado bajo, cuando el cash liquidado no es suficiente, cuando el destino fue modificado recientemente o cuando la solicitud es urgente y el nivel de riesgo aml es medio o alto.

Finalmente, si ninguna de estas condiciones se activa, la solicitud se aprueba automaticamente.

Cada decision se acompaña de un `reason_code` y un valor de `severity`. Esto permite priorizar facilmente los casos que requieren revision manual.

In [2]:
import pandas as pd
from src.motor_decisiones_retiros import construir_motor_decisiones

ruta = "datos/withdrawals.xlsx"

withdrawal_requests = pd.read_excel(ruta, sheet_name="withdrawal_requests")
account_snapshot = pd.read_excel(ruta, sheet_name="account_snapshot")
destination_registry = pd.read_excel(ruta, sheet_name="destination_registry")

decisiones = construir_motor_decisiones(
    withdrawal_requests,
    account_snapshot,
    destination_registry,
)

decisiones.head()

,request_id,decision,reason_code,severity
90,W100101,HOLD,DEST_CHANGED_RECENTLY,45
105,W100029,HOLD,INSUFFICIENT_AVAILABLE_AFTER_BUFFER,55
96,W100109,HOLD,INSUFFICIENT_AVAILABLE_AFTER_BUFFER,55
60,W100035,APPROVE,None,0
24,W100105,REJECT,KYC_NOT_VERIFIED,100


In [3]:
# filtrar solo retiros que requieren revision
review = decisiones[decisiones["decision"] == "HOLD"]

# ordenar por severidad descendente
review = review.sort_values("severity", ascending=False)

# guardar archivo para revision manual
review.to_csv("salidas/review_retiros_hold.csv", index=False)

review.head()

,request_id,decision,reason_code,severity
82,W100054,HOLD,INSUFFICIENT_SETTLED_AFTER_BUFFER,65
108,W100052,HOLD,INSUFFICIENT_SETTLED_AFTER_BUFFER,65
39,W100022,HOLD,INSUFFICIENT_SETTLED_AFTER_BUFFER,65
99,W100032,HOLD,INSUFFICIENT_SETTLED_AFTER_BUFFER,65
30,W100012,HOLD,INSUFFICIENT_SETTLED_AFTER_BUFFER,65


### idea para escalar el proceso usando agentes

Una posible evolucion de este sistema seria convertir el motor de decisiones en un servicio automatizado que funcione como parte de un flujo operativo mas amplio.

Por ejemplo:

1. un sistema de ingestion recibe nuevas solicitudes de retiro en tiempo real.
2. un microservicio ejecuta el motor de decisiones automaticamente.
3. las solicitudes aprobadas se envian directamente al sistema de pagos.
4. las solicitudes en estado HOLD se envian a una cola de revision manual priorizada por severity.
5. un panel operativo permite a los analistas revisar estos casos y tomar decisiones.

Este flujo podria implementarse utilizando herramientas como:

- **Python + FastAPI** para exponer el motor de decisiones como API
- **PostgreSQL** para almacenar solicitudes y decisiones
- **Redis / Kafka** para manejar eventos o colas de procesamiento
- **Airflow o Prefect** para orquestar procesos batch si fuera necesario
- **un dashboard en Streamlit o Metabase** para revision manual

Con esta arquitectura el sistema podria escalar facilmente a miles de solicitudes diarias manteniendo controles de riesgo y trazabilidad de las decisiones.